
# Project Scenario:

An international firm that is looking to expand its business in different countries across the world has recruited you. You have been hired as a junior Data Engineer and are tasked with creating an automated script that can extract the list of all countries in order of their GDPs in billion USDs (rounded to 2 decimal places), as logged by the International Monetary Fund (IMF). Since IMF releases this evaluation twice a year, this code will be used by the organization to extract the information as it is updated.

The required data seems to be available on the URL mentioned below:
https://web.archive.org/web/20230902185326/https://en.wikipedia.org/wiki/List_of_countries_by_GDP_%28nominal%29

The required information needs to be made accessible as a CSV file Countries_by_GDP.csv as well as a table Countries_by_GDP in a database file World_Economies.db with attributes Country and GDP_USD_billion.

Your boss wants you to demonstrate the success of this code by running a query on the database table to display only the entries with more than a 100 billion USD economy. Also, you should log in a file with the entire process of execution named etl_project_log.txt.

You must create a Python code 'etl_project_gdp.py' that performs all the required tasks.

## Objectives

You have to complete the following tasks for this project

* Write a data extraction function to retrieve the relevant information from the required URL.
* Transform the available GDP information into 'Billion USD' from 'Million USD'.
* Load the transformed information to the required CSV file and as a database file.
* Run the required query on the database.
* Log the progress of the code with appropriate timestamps.




In [1]:
from bs4 import BeautifulSoup
import datetime as dt
import numpy as np
import pandas as pd
from re import sub
import requests
import sqlite3

# Setup

I could not use the URL provided as the Archive.org site is blocked by my provider. So I downloaded the website as HTML only and used the file. So rather than use the requests library to download the site text, I use a file from disk instead (called gdp.html).

In [2]:
csv_target_name = "Countries_by_GDP.csv"
database_target_name = "Wordl_Economies.db"
db_table_target_name = "Countries_by_GDP"
data_columns = ["Country", "GDP_USD_million"]
original_data_file_id = "gdp.html"
original_data_url = "https://web.archive.org/web/20230902185326/https://en.wikipedia.org/wiki/List_of_countries_by_GDP_%28nominal%29"
log_file_target = "etl_project_log.txt"

In [3]:
def get_data_local_file(local_file):
    with open("gdp.html") as fp:
        data = BeautifulSoup(fp, "html.parser")
    return data

def get_data_website(target_url):
    html_page = requests.get(target_url).text
    return BeautifulSoup(html_page, "html.parser")

# Extract   

In [4]:
def extract(data, data_columns):
    # Target_table_position refers to the location of the table in regard to the other tables found in the document
    target_table_position = 2
    # Target_row_length refers to the number of columns a table row should have if in the right table 
    target_row_length = 4

    # Dataframe to accumulate all the readings
    gdp_df = pd.DataFrame(columns = data_columns)

    table_rows = data.find_all("tbody")[target_table_position].find_all("tr")
    for row in table_rows: 
        standard_cells = row.find_all("td")
        # Check correct number of columns, and check that there is a hyperlink, this is what contains the country name. The first target row 
        # does not contain a hyperlink as it's a stand in for the world total, not wanted in this project
        if len(standard_cells) == target_row_length and standard_cells[0].a is not None:
            entry = {
                "Country": standard_cells[0].a.text,
                "GDP_USD_million": standard_cells[1].text
            }
            gdp_df = pd.concat([gdp_df, pd.DataFrame(entry, index=[0])])

    return gdp_df

# Transform

In [5]:
def transform(data):
    # Transform the dataset to remove rows that do not have numbers in the GDP USD million column
    # Remove commas
    data["GDP_USD_million"] = data["GDP_USD_million"].str.replace(",", "")
    # Remove non-numeric entries, this includes -na and the year when both value and year were given in same entry, e.g. 12348(2024)
    data["GDP_USD_million"] = data["GDP_USD_million"].str.extract(r"^(\d*)")
    # Remove empty values
    data = data[data["GDP_USD_million"].astype(bool)]

    # Transform the GDP USD million column to integers from string
    data["GDP_USD_million"] = data["GDP_USD_million"].astype(int)

    # Transform the GDP USB million column to billions, assuming American billions
    data["GDP_USD_billion"] = data["GDP_USD_million"].div(1000)
    data = data.drop(['GDP_USD_million'], axis=1)

    return data

# Load

In [6]:
def load(data, csv_target_name, database_target_name, db_table_target_name):
    # Write the CSV file
    data.to_csv(csv_target_name, index=False)    

    # Connect to the database and write the table, replacing if already there
    conn = sqlite3.connect(database_target_name)
    data.to_sql(db_table_target_name, conn, if_exists='replace', index=False)
    conn.close()

# Logging

In [7]:
def log_progress(message, log_file_target): 
    timestamp_format = '%Y-%h-%d-%H:%M:%S' # Year-Monthname-Day-Hour-Minute-Second 
    now = dt.datetime.now(tz=None) # get current timestamp 
    timestamp = now.strftime(timestamp_format)
    with open(log_file_target,"a") as f: 
        f.write(timestamp + ' : ' + message + '\n')

# Querying

In [8]:

def run_query(query, conn):
    result = pd.read_sql(query, conn)
    print(result)

# Execution

In [9]:
log_progress("Starting the extraction", log_file_target)
extracted_data = extract(get_data_local_file(original_data_file_id), data_columns)
log_progress("Completed the extraction", log_file_target)

log_progress("Starting the transformation", log_file_target)
transformed_data = transform(extracted_data)
log_progress("Completed the transformation", log_file_target)

log_progress("Starting the loading", log_file_target)
load(transformed_data,csv_target_name, database_target_name, db_table_target_name)
log_progress("Completed the loading", log_file_target)

log_progress("Staring to query the database", log_file_target)
query_string = f"SELECT * FROM {db_table_target_name} WHERE GDP_USD_billion > 100"
conn = sqlite3.connect(database_target_name)
run_query(query_string, conn)
log_progress("Completed the query of the database", log_file_target)

           Country  GDP_USD_billion
0    United States        32383.920
1            China        20851.593
2          Germany         5452.858
3            Japan         4379.253
4   United Kingdom         4264.794
..             ...              ...
76       Venezuela          111.303
77      Luxembourg          110.417
78      Costa Rica          109.931
79       Lithuania          105.907
80         Belarus          102.042

[81 rows x 2 columns]
